# Rowan Qwen 1.7B Demo Training

This notebook trains two LoRA adapters for the `Love at Dusk` Rowan demo:

- Text generation adapter from `datasets/rowan_ashford_sft.jsonl`
- Reward/score adapter from `datasets/rowan_ashford_reward_demo.jsonl`

Before running: set Colab to `Runtime -> Change runtime type -> GPU`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Project Setup

Option A is easiest if this repo is on GitHub. Option B works if you upload a zip of the repo to Google Drive. Run only one option.


In [ ]:
# Option A: clone from GitHub. Replace this URL with your repo URL, then run this cell.
REPO_URL = ''  # e.g. 'https://github.com/yourname/isekai.git'

if REPO_URL:
    !rm -rf /content/isekai
    !git clone $REPO_URL /content/isekai
else:
    print('Set REPO_URL, or skip this cell and use Option B.')


In [ ]:
# Option B: unzip a repo archive from Drive. Upload isekai.zip to MyDrive first.
ZIP_PATH = '/content/drive/MyDrive/isekai.zip'

import os
if os.path.exists(ZIP_PATH):
    !rm -rf /content/isekai
    !unzip -q $ZIP_PATH -d /content
    print('Unzipped repo archive.')
else:
    print(f'No archive found at {ZIP_PATH}. Use Option A, or upload isekai.zip to Drive.')


In [ ]:
%cd /content/isekai
!pwd
!ls -la


## Install Dependencies

The local repo uses Hugging Face Transformers plus PEFT LoRA training.


In [ ]:
!pip install -U transformers peft accelerate safetensors


## Model Path

If `./qwen3-1.7b` is included in the repo/archive, leave this as-is. Otherwise set `BASE_MODEL` to a Hugging Face model id or a Drive path containing the downloaded Qwen model.


In [ ]:
import os, torch

BASE_MODEL = './qwen3-1.7b'
if not os.path.exists(BASE_MODEL):
    print(f'{BASE_MODEL} not found. Set BASE_MODEL to your downloaded Qwen path or HF model id.')

print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## Validate Datasets


In [ ]:
import json
from pathlib import Path

for path in ['datasets/rowan_ashford_sft.jsonl', 'datasets/rowan_ashford_reward_demo.jsonl']:
    rows = []
    with Path(path).open() as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    print(path, len(rows), 'rows')


## Train Text Generation Adapter


In [ ]:
!python train_text_model.py \
  --model-path $BASE_MODEL \
  --train-file datasets/rowan_ashford_sft.jsonl \
  --output-dir models/rowan-qwen3-1.7b-sft \
  --epochs 3 \
  --batch-size 1 \
  --grad-accum 8


## Train Reward Adapter


In [ ]:
!python train_reward_model.py \
  --model-path $BASE_MODEL \
  --train-file datasets/rowan_ashford_reward_demo.jsonl \
  --output-dir models/rowan-qwen3-1.7b-reward \
  --epochs 5 \
  --batch-size 1 \
  --grad-accum 8


## Save Adapters To Drive


In [ ]:
!mkdir -p /content/drive/MyDrive/isekai-rowan-models
!cp -r models/rowan-qwen3-1.7b-sft /content/drive/MyDrive/isekai-rowan-models/
!cp -r models/rowan-qwen3-1.7b-reward /content/drive/MyDrive/isekai-rowan-models/
!tar -czf /content/drive/MyDrive/isekai-rowan-models/rowan-qwen3-1.7b-adapters.tar.gz models/rowan-qwen3-1.7b-sft models/rowan-qwen3-1.7b-reward
!ls -lh /content/drive/MyDrive/isekai-rowan-models


## Quick Inference Smoke Test

This loads the text adapter and asks Rowan for one short reply.


In [ ]:
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM
import torch

adapter_path = 'models/rowan-qwen3-1.7b-sft'
tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
model = AutoPeftModelForCausalLM.from_pretrained(adapter_path, trust_remote_code=True).eval()
if torch.cuda.is_available():
    model = model.to('cuda')

messages = [
    {'role': 'system', 'content': 'You are Rowan Ashford at The Last Light. Reply briefly, guardedly, and in character.'},
    {'role': 'user', 'content': 'I noticed you kept the cracked mug for me again.'},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt')
if torch.cuda.is_available():
    inputs = inputs.to('cuda')
with torch.inference_mode():
    output = model.generate(inputs, max_new_tokens=120, temperature=0.7, top_p=0.9, do_sample=True, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(output[0, inputs.shape[-1]:], skip_special_tokens=True))
